## 1. Setup

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[1]   # Desktop/Materias/ML/Final Project
W4_SRC = PROJECT_ROOT / "workshops" / "w4" / "src"
for _p in (str(W4_SRC),):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from constants import CHECKPOINTS_DIR, RUNS_DIR, FIGURES_DIR, SEED
from autoencoder.data_loader import load_w1_data, filter_normal
from autoencoder.autoencoder_vae import VAE, VAEConfig, ELBOLoss
from train.train_vae import VAETrainer, save_checkpoint, load_checkpoint, build_loader, set_seed
from evaluate.evaluate_vae import VAEEvaluator
from train.train_ae import load_checkpoint as ae_load_checkpoint

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 2. Data & preprocessor

In [ ]:
df, y_all, features, categorical_cols, numeric_cols, split = load_w1_data()
fit_idx, stop_idx = filter_normal(split, y_all)

# Load the pre-fitted preprocessor from ae_best.pt (do NOT refit)
_, _, prep, _, _ = ae_load_checkpoint(CHECKPOINTS_DIR / "ae_best.pt", device)
print(f"Preprocessor loaded  |  numeric_dim={prep.numeric_dim}  input_dim={prep.input_dim}")

x_num_fit,  x_cat_fit  = prep.transform(df.iloc[fit_idx])
x_num_stop, x_cat_stop = prep.transform(df.iloc[stop_idx])

test_idx = split["test"]
df_test  = df.iloc[test_idx].reset_index(drop=True)
y_test   = y_all[test_idx]        # numpy array
x_num_test, x_cat_test = prep.transform(df_test)

print(f"fit_normal  : {len(x_num_fit):,}")
print(f"stop_normal : {len(x_num_stop):,}")
print(f"test        : {len(x_num_test):,}  (churn={y_test.sum():,})")

## 3. Train beta sweep {0.5, 1.0, 4.0}

In [ ]:
BETAS = [0.5, 1.0, 4.0]
vae_models    = {}
vae_histories = {}

for beta in BETAS:
    print(f"\n{'='*55}")
    print(f"  Training VAE  beta={beta}")
    print(f"{'='*55}")
    set_seed(SEED)
    config = VAEConfig(
        numeric_dim       = prep.numeric_dim,
        cat_cardinalities = prep.cat_cardinalities,
        z_dim             = 16,
        beta              = beta,
        kl_anneal_epochs  = 10,
        max_epochs        = 50,
        patience          = 5,
    )
    model   = VAE(config).to(device)
    trainer = VAETrainer(model, config, device)
    history = trainer.fit(
        build_loader(x_num_fit,  x_cat_fit,  config.batch_size, shuffle=True),
        build_loader(x_num_stop, x_cat_stop, config.batch_size, shuffle=False),
    )
    ckpt_path = CHECKPOINTS_DIR / f"vae_beta{beta}.pt"
    save_checkpoint(ckpt_path, model, config, prep, history, trainer.best_epoch)
    history.to_csv(RUNS_DIR / f"vae_beta{beta}_history.csv", index=False)

    vae_models[beta]    = model
    vae_histories[beta] = history
    print(f"  Saved -> {ckpt_path.name}  |  best_epoch={trainer.best_epoch}")

## 4. Training curves per beta

In [ ]:
fig, axes = plt.subplots(len(BETAS), 4, figsize=(18, 4 * len(BETAS)))
fig.suptitle("VAE training curves by beta", fontsize=14)

for row, beta in enumerate(BETAS):
    h = vae_histories[beta]

    axes[row, 0].plot(h["epoch"], h["train_loss"], label="train")
    axes[row, 0].plot(h["epoch"], h["val_loss"],   label="val")
    axes[row, 0].set_title(f"beta={beta}  total ELBO")
    axes[row, 0].legend(fontsize=8)

    axes[row, 1].plot(h["epoch"], h["val_recon"], label="recon")
    axes[row, 1].plot(h["epoch"], h["val_kl"],    label="KL")
    axes[row, 1].set_title(f"beta={beta}  val recon / KL")
    axes[row, 1].legend(fontsize=8)

    axes[row, 2].plot(h["epoch"], h["beta_t"])
    axes[row, 2].axhline(beta, linestyle="--", color="gray", linewidth=0.8)
    axes[row, 2].set_title(f"beta={beta}  KL annealing")

    axes[row, 3].plot(h["epoch"], h["active_units"])
    axes[row, 3].set_ylim(0, 17)
    axes[row, 3].set_title(f"beta={beta}  active units")

for ax in axes.flat:
    ax.set_xlabel("epoch")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "vae_training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

## 5. Comparison table across beta values

In [ ]:
from sklearn.metrics import silhouette_score

rows = []

for beta in BETAS:
    model = vae_models[beta]
    h     = vae_histories[beta]
    ev    = VAEEvaluator(model, prep, device)

    sm_val = ev.summary_metrics(x_num_stop, x_cat_stop)
    _, _, mu_val, _ = ev.predict(x_num_stop, x_cat_stop)
    au = int((mu_val.var(dim=0) > 0.01).sum())

    sil = ev.latent_separability(x_num_test, x_cat_test, y_test, seed=SEED)

    rows.append({
        "beta":              beta,
        "val_recon":         float(h["val_recon"].iloc[-1]),
        "val_kl":            float(h["val_kl"].iloc[-1]),
        "mse_per_element":   float(h["val_mse_per_element"].iloc[-1]),
        "cat_acc_city":      sm_val["cat_accuracy"].get("city", float("nan")),
        "active_units":      au,
        "silhouette_test":   sil,
    })

comp_df = pd.DataFrame(rows)
print(comp_df.to_string(index=False))

## 6. Choose vae_best

Lower beta: better reconstruction, weaker regularisation.
Higher beta: more structured latent space (higher silhouette), worse reconstruction.
beta=1.0 is the canonical VAE. Adjust `BEST_BETA` if the table suggests otherwise.

In [ ]:
BEST_BETA = 1.0   # adjust after inspecting the table above

import shutil
best_src = CHECKPOINTS_DIR / f"vae_beta{BEST_BETA}.pt"
best_dst = CHECKPOINTS_DIR / "vae_best.pt"
shutil.copy(best_src, best_dst)
print(f"vae_best.pt -> copy of vae_beta{BEST_BETA}.pt")

vae_best, cfg_best, _, _, _ = load_checkpoint(best_dst, device)
ev_best = VAEEvaluator(vae_best, prep, device)

## 7. Test evaluation of vae_best

In [ ]:
# Per-class reconstruction errors
num_out_t, cat_logits_t, mu_t, logvar_t = ev_best.predict(x_num_test, x_cat_test)

_elbo = ELBOLoss()
_, recon_i, _, _ = _elbo(
    num_out_t, x_num_test, cat_logits_t, x_cat_test,
    mu_t, logvar_t, beta=1.0, reduction="none",
)
vae_errors = recon_i.numpy()

print(f"MSE per element (test): {ev_best.summary_metrics(x_num_test, x_cat_test)['mse_numeric']:.6f}")
print()
print(ev_best.metrics_by_class(vae_errors, y_test).to_string(index=False))

In [ ]:
# Top-10 hardest features
vae_pfm = ev_best.per_feature_metrics(x_num_test, x_cat_test)
print("Top-10 hardest features:")
print(vae_pfm.nlargest(10, "mse").to_string(index=False))

In [ ]:
# Sample panel: 8 test clients — original vs VAE reconstruction
import random
panel_idx = random.Random(SEED).sample(range(len(x_num_test)), 8)
x_num_p   = x_num_test[panel_idx]
x_cat_p   = x_cat_test[panel_idx]
df_orig_p = df_test.iloc[panel_idx].reset_index(drop=True)

panel = ev_best.sample_panel(x_num_p, x_cat_p, df_orig_p)
print("=== Reconstruction panel (first 8 numeric cols) ===")
print(panel[prep.numeric_cols_nc[:8]].to_string())

In [ ]:
# Difference heatmap
vae_diff = ev_best.difference_matrix(x_num_test, x_cat_test)

rng_h = np.random.default_rng(SEED)
idx_h = rng_h.choice(len(vae_diff), 2000, replace=False)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(vae_diff[idx_h].T, aspect="auto", cmap="hot", vmin=0, vmax=3)
ax.set_title(f"VAE beta={BEST_BETA} |x - x_hat| (standardised)")
ax.set_xlabel("sample"); ax.set_ylabel("feature")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "vae_difference_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()

## 8. KL per latent dimension (vae_best)

In [ ]:
_, _, mu_test, logvar_test = ev_best.predict(x_num_test, x_cat_test)
kl_dims = ev_best.kl_per_dimension(mu_test, logvar_test)

fig, ax = plt.subplots(figsize=(10, 4))
dims = np.arange(len(kl_dims))
ax.bar(dims, kl_dims, color=np.where(kl_dims > 0.1, "steelblue", "lightgray"))
ax.axhline(0.1, linestyle="--", color="red", linewidth=0.8, label="active threshold (0.1)")
ax.set_xlabel("latent dimension")
ax.set_ylabel("mean KL")
ax.set_title(f"KL per latent dimension -- VAE beta={BEST_BETA}")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "vae_kl_per_dim.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Active dims (KL>0.1): {int((kl_dims > 0.1).sum())} / {len(kl_dims)}")

## 9. Generative demo — 8 clients sampled from N(0, I)

The VAE's structured prior allows sampling directly from the latent space.

In [ ]:
set_seed(SEED)
gen_df = ev_best.generate_samples(8)
print("Generated clients (59 columns):")
print(gen_df.to_string())

fig, ax = plt.subplots(figsize=(14, 3))
ax.imshow(gen_df[prep.numeric_cols_nc[:20]].to_numpy().T, aspect="auto", cmap="viridis")
ax.set_xlabel("generated client")
ax.set_ylabel("numeric feature (first 20)")
ax.set_title(f"VAE beta={BEST_BETA} -- 8 clients sampled from N(0, I)")
ax.set_xticks(range(8)); ax.set_xticklabels([f"g{i}" for i in range(8)])
plt.tight_layout()
plt.savefig(FIGURES_DIR / "vae_generative_demo.png", dpi=120, bbox_inches="tight")
plt.show()